# SegRNN — Stage 2c: follow-up architecture ideas, revised variants, and a reservoir-computing test

Third, independent notebook — mounts Drive and clones/pulls the repo
itself, same as the first two. Five new architecture strands (each
following up on something already found in `colab_runner.ipynb` /
`colab_runner_stage2b.ipynb`), two revised variants built after two of
those underperformed, and three parts testing whether turning SegRNN's
own recurrent cell into a frozen Echo State Network reservoir works —
untested anywhere in the paper or this project:

- **Part 1** — `SegRNNUnshared`: un-share the decode cell but keep the
  encoder unidirectional, isolating one of the two things `SegRNNBidir`
  changed at once (that strand came back neutral, a surprise — this asks
  whether un-sharing alone explains it).
- **Part 2** — `SegRNNPoolContext`: retests `SegRNNAttn`'s bottleneck-
  removal hypothesis with a parameter-free mean/max pool instead of
  learned attention — zero added parameters vs. `SegRNNAttn`'s Q/K/V.
- **Part 3** — `SegRNNWeightTied`: ties the value-embedding and predict
  weights (exact shape-transposes of each other) — capacity reduction
  via structure, the same direction as `d_model=256` and Huber loss.
- **Part 4** — `SegRNNLinearShortcut`: a DLinear-style linear path
  blended with the RNN path via a learned per-channel gate. Regressed —
  see Part 7 for a revised version.
- **Part 5** — `SegRNNLayerNorm`: `LayerNorm` after the value embedding —
  near-zero parameters, an optimization aid rather than added richness.
  Regressed — see Part 6 for a revised version.
- **Part 6** — `SegRNNLayerNormHidden`: same near-zero-cost idea as Part
  5, repositioned — normalizes `h_n` (the actual bottleneck) right before
  decoding, instead of the input embedding.
- **Part 7** — post-hoc ensemble of independently-trained `SegRNN` and
  `DLinear` (already in `models/`, a baseline this project already
  reconstructed): average their *predictions*, not their metrics — same
  logic as the seed-ensembling strand, across model families instead of
  seeds. Avoids Part 4's joint-training blend-gate interference, and
  uses real DLinear (trend/seasonal decomposition) rather than one plain
  linear layer.
- **Part 8** — `SegRNNReservoir`, naive frozen ablation: freeze SegRNN's
  recurrent cell (`--rnn_type rnn`) at PyTorch's raw default
  initialization, no reservoir tuning. A control for Part 9.
  Every other strand that adds *trained* capacity has regressed or landed
  neutral in this project; freezing the recurrent core entirely (training
  only the value embedding, positional embeddings, and predict head — a
  more radical version of the same "less trained capacity" lever behind
  `d_model=256`, Huber loss, and pool context) is untested territory.
- **Part 9** — `SegRNNReservoir`, proper Echo State Network init: same
  frozen cell, but the recurrent weight matrix's spectral radius is
  rescaled to the standard ESN range (~0.9) before freezing — the
  "echo state property" real reservoir computing relies on, which naive
  random init isn't tuned for. Compares directly against Part 8 to see
  whether that tuning matters.
- **Part 10** — spectral radius sweep (`--reservoir_spectral_radius` in
  `{0.5, 0.9, 0.99, 1.1}`, 2 horizons to control cost): tests whether
  results are actually sensitive to the echo-state property the way ESN
  theory predicts, or whether Part 9's result (if any) is incidental.

Each tested only against `RECON_BASELINE` (hardcoded from
`results/runs.csv`), independent of every other strand — same reasoning
as every prior notebook. ~36 new training runs total; Parts don't depend
on each other.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi

In [ ]:
import os
REPO = "https://github.com/amitzr/SegRNN.git"
if not os.path.exists('/content/proj'):
    !git clone $REPO /content/proj
%cd /content/proj
!git pull

In [ ]:
import os
if not os.path.exists('/content/proj/dataset'):
    os.symlink('/content/drive/MyDrive/ts-project/dataset', '/content/proj/dataset')
!ls -la /content/proj/dataset | head
!pip install -q -r requirements.txt

## Setup: shared constants, training runner, plotting

Same `run_horizon`/plotting pattern as the other two notebooks.

In [ ]:
import os, sys, re, csv, subprocess, datetime, statistics
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

sys.path.insert(0, os.getcwd())  # make scripts.*, utils.*, data_provider.* importable

HORIZONS = [96, 192, 336, 720]

# Reconstruction baseline (SegRNN, d_model=512, seed=2024), from results/runs.csv --
# the same comparison point every Stage 2 strand in this project uses.
RECON_BASELINE = {
    'mse': {96: 0.3510, 192: 0.3925, 336: 0.4233, 720: 0.4657},
    'mae': {96: 0.3925, 192: 0.4142, 336: 0.4327, 720: 0.4719},
}

os.makedirs('results/figures', exist_ok=True)


def run_horizon(model, pred_len, seed=None, d_model=512, revin=None, power_transform=None,
                 loss=None, save_preds=None, **extra_flags):
    """Launch run_longExp.py for one (model, horizon), stream its output
    live, and parse the final 'mse:X, mae:Y, ms/sample:Z' line it prints.
    Returns (mse, mae, ms_per_sample). extra_flags passes through arbitrary
    --flag value pairs (e.g. pool_type='max', rnn_type='rnn',
    reservoir_scale_init=0) -- repeated flags with a different value than
    the base cmd list (e.g. rnn_type) simply overwrite it, argparse
    last-value-wins."""
    model_id = (f'ETTh1_720_{pred_len}'
                + (f'_seed{seed}' if seed is not None else '')
                + (f'_dm{d_model}' if d_model != 512 else '')
                + (f'_revin{revin}' if revin is not None else ''))
    cmd = [
        'python', '-u', 'run_longExp.py',
        '--is_training', '1', '--model_id', model_id, '--model', model, '--data', 'ETTh1',
        '--root_path', './dataset/', '--data_path', 'ETTh1.csv',
        '--features', 'M', '--seq_len', '720', '--pred_len', str(pred_len),
        '--seg_len', '24', '--enc_in', '7', '--d_model', str(d_model),
        '--dropout', '0.1', '--rnn_type', 'gru', '--dec_way', 'pmf', '--channel_id', '1',
        '--train_epochs', '30', '--patience', '5',
        '--itr', '1', '--batch_size', '64', '--learning_rate', '0.0003',
    ]
    if seed is not None:
        cmd += ['--random_seed', str(seed)]
    if revin is not None:
        cmd += ['--revin', str(revin)]
    if power_transform is not None:
        cmd += ['--power_transform', str(power_transform)]
    if loss is not None:
        cmd += ['--loss', loss]
    if save_preds is not None:
        cmd += ['--save_preds', str(save_preds)]
    for flag, val in extra_flags.items():
        cmd += [f'--{flag}', str(val)]

    print(f'\n{"="*70}\n{model}  H={pred_len}' + (f'  {extra_flags}' if extra_flags else '') + f'\n{"="*70}')
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in proc.stdout:
        print(line, end='')
        tail.append(line)
        if len(tail) > 5:
            tail.pop(0)
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f'{model} H={pred_len} failed (exit {proc.returncode}) -- see output above')

    m = re.search(r'mse:([\d.]+), mae:([\d.]+), ms/sample:([\d.]+)', ''.join(tail))
    if not m:
        raise RuntimeError(f'Could not find mse/mae/ms-per-sample in output for {model} H={pred_len}')
    return float(m.group(1)), float(m.group(2)), float(m.group(3))


# dataviz-validated categorical palette, fixed order, this notebook's own labels
COLORS = {
    'Reconstruction': '#008300',
    'Unshared Decode': '#2a78d6',
    'Pool Context': '#9c2c8f',
    'Weight Tied': '#a35a00',
    'Linear Shortcut': '#0a8fa3',
    'LayerNorm': '#e34948',
    'LayerNorm (hidden)': '#4a3aa7',
    'SegRNN+DLinear Ensemble': '#c9a227',
    'Naive Frozen': '#5a7d2a',
    'ESN Reservoir': '#b34d8c',
}
INK_PRIMARY, INK_SECONDARY, INK_MUTED = '#0b0b0b', '#52514e', '#898781'
GRIDLINE, BASELINE_AXIS, SURFACE = '#e1e0d9', '#c3c2b7', '#fcfcfb'


def plot_metric(metric_name, series, save_path=None):
    """series: list of (label, {horizon: value}), in display order.
    Always creates a brand-new figure."""
    n_series = len(series)
    x = np.arange(len(HORIZONS))
    group_width = 0.8
    bar_width = group_width / n_series

    fig, ax = plt.subplots(figsize=(9, 5.5), facecolor=SURFACE)
    ax.set_facecolor(SURFACE)
    for i, (label, values) in enumerate(series):
        offsets = x - group_width / 2 + bar_width * (i + 0.5)
        heights = [values[h] for h in HORIZONS]
        bars = ax.bar(offsets, heights, width=bar_width * 0.9, color=COLORS[label],
                       label=label, edgecolor=SURFACE, linewidth=0.5)
        for bar, h in zip(bars, heights):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f'{h:.3f}',
                     ha='center', va='bottom', fontsize=7.5, color=INK_PRIMARY)

    ax.set_xticks(x)
    ax.set_xticklabels([f'H={h}' for h in HORIZONS], color=INK_SECONDARY)
    ax.set_ylabel(metric_name.upper(), color=INK_SECONDARY)
    ax.set_title(f'SegRNN on ETTh1 — {metric_name.upper()}', color=INK_PRIMARY, fontsize=13, loc='left')
    ax.yaxis.grid(True, color=GRIDLINE, linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)
    for spine in ('top', 'right', 'left'):
        ax.spines[spine].set_visible(False)
    ax.spines['bottom'].set_color(BASELINE_AXIS)
    ax.tick_params(axis='both', which='both', length=0, colors=INK_MUTED)
    ax.legend(frameon=False, loc='upper left', bbox_to_anchor=(0, 1.14),
              ncol=n_series, fontsize=9, labelcolor=INK_SECONDARY)

    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=200, facecolor=SURFACE)
        print(f'saved {save_path}')
    plt.show()


def make_table(metric_name, series):
    rows = []
    for h in HORIZONS:
        row = {'Horizon': h}
        for label, values in series:
            row[label] = round(values[h], 4)
        rows.append(row)
    print(f'\n{metric_name.upper()}')
    return pd.DataFrame(rows).set_index('Horizon')

## Part 1 — un-shared decode cell (`SegRNNUnshared`)

Encoder stays unidirectional (identical to `models/SegRNN.py`); only the decode cell is un-shared, given its own independently-learned GRU. Isolates one of the two things `SegRNNBidir` changed at once — see `models/SegRNNUnshared.py`'s docstring for the full logic.

In [ ]:
unshared_results = {}
for h in HORIZONS:
    unshared_results[h] = run_horizon('SegRNNUnshared', h)

unshared_series_mse = [
    ('Reconstruction', RECON_BASELINE['mse']),
    ('Unshared Decode', {h: unshared_results[h][0] for h in HORIZONS}),
]
unshared_series_mae = [
    ('Reconstruction', RECON_BASELINE['mae']),
    ('Unshared Decode', {h: unshared_results[h][1] for h in HORIZONS}),
]
display(make_table('mse', unshared_series_mse))
display(make_table('mae', unshared_series_mae))
plot_metric('mse', unshared_series_mse, save_path='results/figures/unshared_comparison_mse.png')
plot_metric('mae', unshared_series_mae, save_path='results/figures/unshared_comparison_mae.png')

## Part 2 — parameter-free pooling context (`SegRNNPoolContext`)

Mean-pools (or max-pools, `--pool_type`) all `n` encoder segment states into a context vector added to `h_n` — zero new learnable parameters, unlike `SegRNNAttn`'s Q/K/V projections. Retests the bottleneck-removal hypothesis at essentially zero added capacity.

In [ ]:
poolcontext_results = {}
for h in HORIZONS:
    poolcontext_results[h] = run_horizon('SegRNNPoolContext', h)

poolcontext_series_mse = [
    ('Reconstruction', RECON_BASELINE['mse']),
    ('Pool Context', {h: poolcontext_results[h][0] for h in HORIZONS}),
]
poolcontext_series_mae = [
    ('Reconstruction', RECON_BASELINE['mae']),
    ('Pool Context', {h: poolcontext_results[h][1] for h in HORIZONS}),
]
display(make_table('mse', poolcontext_series_mse))
display(make_table('mae', poolcontext_series_mae))
plot_metric('mse', poolcontext_series_mse, save_path='results/figures/poolcontext_comparison_mse.png')
plot_metric('mae', poolcontext_series_mae, save_path='results/figures/poolcontext_comparison_mae.png')

## Part 3 — weight-tied value embedding / predict head (`SegRNNWeightTied`)

Value embedding and predict head share one weight matrix (exact shape-transposes of each other) — a structural capacity reduction, the same direction as `d_model=256` and Huber loss, applied architecturally instead of via a hyperparameter.

In [ ]:
weighttied_results = {}
for h in HORIZONS:
    weighttied_results[h] = run_horizon('SegRNNWeightTied', h)

weighttied_series_mse = [
    ('Reconstruction', RECON_BASELINE['mse']),
    ('Weight Tied', {h: weighttied_results[h][0] for h in HORIZONS}),
]
weighttied_series_mae = [
    ('Reconstruction', RECON_BASELINE['mae']),
    ('Weight Tied', {h: weighttied_results[h][1] for h in HORIZONS}),
]
display(make_table('mse', weighttied_series_mse))
display(make_table('mae', weighttied_series_mae))
plot_metric('mse', weighttied_series_mse, save_path='results/figures/weighttied_comparison_mse.png')
plot_metric('mae', weighttied_series_mae, save_path='results/figures/weighttied_comparison_mae.png')

## Part 4 — DLinear-style linear shortcut (`SegRNNLinearShortcut`)

A parallel `Linear(seq_len -> pred_len)` path, bypassing the GRU entirely, blended with the RNN path via a learned per-channel gate (sigmoid, initialized at 0.5 -- balanced). Motivated directly by the paper's own text: DLinear beats several Transformer baselines in the univariate setting.

In [ ]:
linearshortcut_results = {}
for h in HORIZONS:
    linearshortcut_results[h] = run_horizon('SegRNNLinearShortcut', h)

linearshortcut_series_mse = [
    ('Reconstruction', RECON_BASELINE['mse']),
    ('Linear Shortcut', {h: linearshortcut_results[h][0] for h in HORIZONS}),
]
linearshortcut_series_mae = [
    ('Reconstruction', RECON_BASELINE['mae']),
    ('Linear Shortcut', {h: linearshortcut_results[h][1] for h in HORIZONS}),
]
display(make_table('mse', linearshortcut_series_mse))
display(make_table('mae', linearshortcut_series_mae))
plot_metric('mse', linearshortcut_series_mse, save_path='results/figures/linearshortcut_comparison_mse.png')
plot_metric('mae', linearshortcut_series_mae, save_path='results/figures/linearshortcut_comparison_mae.png')

## Part 5 — LayerNorm after value embedding (`SegRNNLayerNorm`)

`nn.LayerNorm(d_model)` appended to the value embedding -- near-zero added parameters (2*d_model), an optimization aid rather than added information, unlike every strand in this project's "inject information" family.

In [ ]:
layernorm_results = {}
for h in HORIZONS:
    layernorm_results[h] = run_horizon('SegRNNLayerNorm', h)

layernorm_series_mse = [
    ('Reconstruction', RECON_BASELINE['mse']),
    ('LayerNorm', {h: layernorm_results[h][0] for h in HORIZONS}),
]
layernorm_series_mae = [
    ('Reconstruction', RECON_BASELINE['mae']),
    ('LayerNorm', {h: layernorm_results[h][1] for h in HORIZONS}),
]
display(make_table('mse', layernorm_series_mse))
display(make_table('mae', layernorm_series_mae))
plot_metric('mse', layernorm_series_mse, save_path='results/figures/layernorm_comparison_mse.png')
plot_metric('mae', layernorm_series_mae, save_path='results/figures/layernorm_comparison_mae.png')

## Part 6 — LayerNorm on `h_n` instead of the input embedding (`SegRNNLayerNormHidden`)

Revised version of Part 5, which regressed. Same near-zero-cost idea (`nn.LayerNorm(d_model)`, `2*d_model` added parameters), repositioned to normalize the actual bottleneck representation (`h_n`, right before decoding) instead of the input embedding — the placement a Transformer block would use (normalize hidden states, not raw inputs). See `models/SegRNNLayerNormHidden.py`'s docstring.

In [ ]:
layernormhidden_results = {}
for h in HORIZONS:
    layernormhidden_results[h] = run_horizon('SegRNNLayerNormHidden', h)

layernormhidden_series_mse = [
    ('Reconstruction', RECON_BASELINE['mse']),
    ('LayerNorm (hidden)', {h: layernormhidden_results[h][0] for h in HORIZONS}),
]
layernormhidden_series_mae = [
    ('Reconstruction', RECON_BASELINE['mae']),
    ('LayerNorm (hidden)', {h: layernormhidden_results[h][1] for h in HORIZONS}),
]
display(make_table('mse', layernormhidden_series_mse))
display(make_table('mae', layernormhidden_series_mae))
plot_metric('mse', layernormhidden_series_mse, save_path='results/figures/layernormhidden_comparison_mse.png')
plot_metric('mae', layernormhidden_series_mae, save_path='results/figures/layernormhidden_comparison_mae.png')

## Part 7 — post-hoc ensemble of independently-trained `SegRNN` + `DLinear`

Revised version of Part 4, which regressed. Instead of a jointly-trained blend gate on top of one plain linear layer, this trains `SegRNN` and the repo's own `DLinear` (Section V-B2's own comparison point) completely independently — each with `--save_preds 1` — then averages their raw *predictions* (not their metrics) and scores the average. Same "average predictions, not metrics" logic the seed-ensembling strand used, across model families instead of seeds; avoids any joint-training interference between the two paths, and uses real DLinear (moving-average trend + seasonal decomposition, two separate linear layers) rather than one flat `Linear(seq_len, pred_len)`.

In [ ]:
import glob
from utils.metrics import MSE, MAE

def load_preds(model, h):
    model_id = f'ETTh1_720_{h}'
    matches = glob.glob(f'results/{model_id}_{model}_*/pred.npy')
    if not matches:
        raise FileNotFoundError(f'no pred.npy for model={model} H={h} -- rerun with save_preds=1')
    folder = os.path.dirname(matches[0])
    return np.load(os.path.join(folder, 'pred.npy')), np.load(os.path.join(folder, 'true.npy'))

ensemble_results = {}
for h in HORIZONS:
    segrnn_mse, segrnn_mae, _ = run_horizon('SegRNN', h, save_preds=1)
    dlinear_mse, dlinear_mae, _ = run_horizon('DLinear', h, save_preds=1)
    segrnn_preds, trues = load_preds('SegRNN', h)
    dlinear_preds, _ = load_preds('DLinear', h)
    avg_pred = (segrnn_preds + dlinear_preds) / 2
    ensemble_results[h] = (MSE(avg_pred, trues), MAE(avg_pred, trues))
    print(f'H={h}: SegRNN alone MSE={segrnn_mse:.4f}, DLinear alone MSE={dlinear_mse:.4f}, '
          f'Ensemble MSE={ensemble_results[h][0]:.4f}')

ensemble_series_mse = [
    ('Reconstruction', RECON_BASELINE['mse']),
    ('SegRNN+DLinear Ensemble', {h: ensemble_results[h][0] for h in HORIZONS}),
]
ensemble_series_mae = [
    ('Reconstruction', RECON_BASELINE['mae']),
    ('SegRNN+DLinear Ensemble', {h: ensemble_results[h][1] for h in HORIZONS}),
]
display(make_table('mse', ensemble_series_mse))
display(make_table('mae', ensemble_series_mae))
plot_metric('mse', ensemble_series_mse, save_path='results/figures/segrnn_dlinear_ensemble_mse.png')
plot_metric('mae', ensemble_series_mae, save_path='results/figures/segrnn_dlinear_ensemble_mae.png')

## Part 8 — naive frozen reservoir, no ESN tuning (`SegRNNReservoir`, control)

Freezes SegRNN's own recurrent cell (`--rnn_type rnn`, a plain tanh cell — GRU/LSTM gates are meant to be *trained*, freezing them randomly would be a much weaker experiment) at PyTorch's raw default initialization, `--reservoir_scale_init 0`. Only the value embedding, positional embeddings, and predict head remain trainable — roughly 5-10% of the reconstruction's trainable parameters. This is the control for Part 9: does freezing alone do anything, without the reservoir-specific initialization that real ESN theory calls for?

In [ ]:
naivefrozen_results = {}
for h in HORIZONS:
    naivefrozen_results[h] = run_horizon('SegRNNReservoir', h, rnn_type='rnn', reservoir_scale_init=0)

naivefrozen_series_mse = [
    ('Reconstruction', RECON_BASELINE['mse']),
    ('Naive Frozen', {h: naivefrozen_results[h][0] for h in HORIZONS}),
]
naivefrozen_series_mae = [
    ('Reconstruction', RECON_BASELINE['mae']),
    ('Naive Frozen', {h: naivefrozen_results[h][1] for h in HORIZONS}),
]
display(make_table('mse', naivefrozen_series_mse))
display(make_table('mae', naivefrozen_series_mae))
plot_metric('mse', naivefrozen_series_mse, save_path='results/figures/naivefrozen_comparison_mse.png')
plot_metric('mae', naivefrozen_series_mae, save_path='results/figures/naivefrozen_comparison_mae.png')

## Part 9 — proper Echo State Network reservoir (`SegRNNReservoir`)

Same frozen cell as Part 8, but the recurrent weight matrix's spectral radius is rescaled to 0.9 (`--reservoir_scale_init 1`, the default) before freezing — the standard ESN reservoir range for the echo-state property (stable, fading memory), computed once via `torch.linalg.eigvals` and never touched again during training. See `models/SegRNNReservoir.py`'s docstring for why this specific tuning step is what separates "a real reservoir" from "a frozen random layer."

In [ ]:
reservoir_results = {}
for h in HORIZONS:
    reservoir_results[h] = run_horizon('SegRNNReservoir', h, rnn_type='rnn',
                                        reservoir_scale_init=1, reservoir_spectral_radius=0.9)

reservoir_series_mse = [
    ('Reconstruction', RECON_BASELINE['mse']),
    ('ESN Reservoir', {h: reservoir_results[h][0] for h in HORIZONS}),
]
reservoir_series_mae = [
    ('Reconstruction', RECON_BASELINE['mae']),
    ('ESN Reservoir', {h: reservoir_results[h][1] for h in HORIZONS}),
]
display(make_table('mse', reservoir_series_mse))
display(make_table('mae', reservoir_series_mae))
plot_metric('mse', reservoir_series_mse, save_path='results/figures/reservoir_comparison_mse.png')
plot_metric('mae', reservoir_series_mae, save_path='results/figures/reservoir_comparison_mae.png')

print('Naive frozen (raw init) vs. proper ESN init (spectral radius 0.9), MSE:')
for h in HORIZONS:
    print(f"  H={h}: naive={naivefrozen_results[h][0]:.4f}, ESN={reservoir_results[h][0]:.4f}, "
          f"reconstruction={RECON_BASELINE['mse'][h]:.4f}")

## Part 10 — spectral radius sweep (`SegRNNReservoir`)

Sweeps `--reservoir_spectral_radius` across `{0.5, 0.9, 0.99, 1.1}` — below, inside, at the edge of, and above the standard ESN stability range — at 2 horizons (336, 720; same compute-budget reasoning as the first notebook's Part 6 and this project's other narrowed-horizon strands). If ESN theory's echo-state property actually matters here, results should be sensitive to this value, likely degrading past radius=1 (loss of the fading-memory/stability guarantee); if results are flat across all four values, Part 9's outcome (whatever it is) isn't really about reservoir dynamics.

In [ ]:
RADII = [0.5, 0.9, 0.99, 1.1]
RESERVOIR_HORIZONS = [336, 720]

radius_results = {}
for r in RADII:
    for h in RESERVOIR_HORIZONS:
        radius_results[(r, h)] = run_horizon('SegRNNReservoir', h, rnn_type='rnn',
                                              reservoir_scale_init=1, reservoir_spectral_radius=r)

mse_rows = []
mae_rows = []
for h in RESERVOIR_HORIZONS:
    mse_row = {'Horizon': h, 'Reconstruction': RECON_BASELINE['mse'][h]}
    mae_row = {'Horizon': h, 'Reconstruction': RECON_BASELINE['mae'][h]}
    for r in RADII:
        mse_row[f'radius={r}'] = round(radius_results[(r, h)][0], 4)
        mae_row[f'radius={r}'] = round(radius_results[(r, h)][1], 4)
    mse_rows.append(mse_row)
    mae_rows.append(mae_row)

print('MSE by horizon x spectral radius')
display(pd.DataFrame(mse_rows).set_index('Horizon'))
print('\nMAE by horizon x spectral radius')
display(pd.DataFrame(mae_rows).set_index('Horizon'))

## Optional — save results back into the repo

Appends this notebook's rows to `results/runs.csv`. Commit/push left commented out on purpose — review `git status`/`git diff` first.

In [ ]:
RUNS_CSV_HEADER = ['run_id','timestamp','model','dataset','horizon','seq_len','seg_len',
                    'd_model','seed','flags','mse','mae','mase','epoch_time_s','params',
                    'peak_mem_mb','notes']
ts = datetime.datetime.now().isoformat(timespec='seconds')
rows = []

strand_defs = [
    ('unshared_results', 'SegRNNUnshared', 'seg_len=24;d_model=512', 'unshared decode cell, unidirectional encoder'),
    ('poolcontext_results', 'SegRNNPoolContext', 'seg_len=24;d_model=512;pool_type=mean', 'parameter-free pooling context'),
    ('weighttied_results', 'SegRNNWeightTied', 'seg_len=24;d_model=512', 'tied value-embedding/predict weights'),
    ('linearshortcut_results', 'SegRNNLinearShortcut', 'seg_len=24;d_model=512', 'DLinear-style blended shortcut'),
    ('layernorm_results', 'SegRNNLayerNorm', 'seg_len=24;d_model=512', 'LayerNorm after value embedding'),
    ('layernormhidden_results', 'SegRNNLayerNormHidden', 'seg_len=24;d_model=512', 'LayerNorm on h_n instead of input embedding'),
    ('naivefrozen_results', 'SegRNNReservoir', 'seg_len=24;d_model=512;rnn_type=rnn;reservoir_scale_init=0', 'naive frozen recurrent cell, no ESN tuning'),
    ('reservoir_results', 'SegRNNReservoir', 'seg_len=24;d_model=512;rnn_type=rnn;reservoir_scale_init=1;reservoir_spectral_radius=0.9', 'proper ESN reservoir (spectral radius 0.9)'),
]
for var_name, model_name, flags, note in strand_defs:
    if var_name in dir():
        for h, (mse, mae, ms) in eval(var_name).items():
            rows.append([f'{model_name}_ETTh1_{h}_{ts}', ts, model_name, 'ETTh1', h, 720, 24, 512, 2024,
                         flags, mse, mae, '', '', '', '', note])

if 'ensemble_results' in dir():
    for h, (mse, mae) in ensemble_results.items():
        rows.append([f'SegRNN+DLinear_ETTh1_{h}_{ts}', ts, 'SegRNN+DLinear', 'ETTh1', h, 720, 24, 512, 2024,
                     'seg_len=24;d_model=512', mse, mae, '', '', '', '',
                     'post-hoc prediction average of independently-trained SegRNN and DLinear'])

if 'radius_results' in dir():
    for (r, h), (mse, mae, ms) in radius_results.items():
        rows.append([f'SegRNNReservoir_ETTh1_{h}_r{r}_{ts}', ts, 'SegRNNReservoir', 'ETTh1', h, 720, 24, 512, 2024,
                     f'seg_len=24;d_model=512;rnn_type=rnn;reservoir_scale_init=1;reservoir_spectral_radius={r}',
                     mse, mae, '', '', '', '', f'ESN spectral radius sweep: radius={r}'])

existing = pd.read_csv('results/runs.csv') if os.path.exists('results/runs.csv') else pd.DataFrame(columns=RUNS_CSV_HEADER)
new_df = pd.DataFrame(rows, columns=RUNS_CSV_HEADER)
combined = pd.concat([existing, new_df], ignore_index=True)
combined.to_csv('results/runs.csv', index=False)
print(f'appended {len(rows)} rows to results/runs.csv (total {len(combined)})')

!git add results/runs.csv results/figures/
!git status
# review the diff above, then when ready:
# !git commit -m "Update results: unshared decode, pool context, weight tied, linear shortcut, layernorm, layernorm-hidden, segrnn+dlinear ensemble, reservoir"
# !git push origin main